# KYC - field detector training (Kaggle GPU -> ONNX)

Pulls the labelled cards, rectifies them, trains the field detector and
exports it to ONNX for CPU serving. One run, start to finish.

**Before you run:**

1. Settings > Accelerator > **GPU T4 x2**
2. Settings > Internet > **On**
3. Add-ons > Secrets > add **`ROBOFLOW_API_KEY`** (only needed when
   `DATA_SOURCE = "roboflow"`)

The exported `.onnx` lands in `/kaggle/working/models/`. Fetch it with
`python -m scripts.kaggle_run pull`, or download it from the Output tab.


## 1. Configuration - the only cell you edit

In [ ]:
# Where the labelled cards come from:
#   "roboflow"       - pull them here, inside the kernel. Use this when your
#                      own network cannot reach Roboflow's CDN.
#   "kaggle-dataset" - use a dataset you already attached under Add Data.
#
# On "kaggle-dataset" because this project's Roboflow exports are missing from
# Roboflow's own storage: the API advertises them and hands out signed links
# that 404, from any network. The cards were downloaded by hand once and
# uploaded to Kaggle instead, which also takes Roboflow out of the run path.
DATA_SOURCE = "kaggle-dataset"

# --- DATA_SOURCE = "roboflow" ---------------------------------------
RF_WORKSPACE = "westco"
RF_PROJECT   = "idcardsegmentation-gz97d"
# v8 is the newest version, but its yolov8 export file is missing from
# Roboflow's storage - the API still advertises it and hands out a link that
# 404s, from any network. v7 generates on demand and works. Move this back to
# 8 once that version has been regenerated in the dashboard.
RF_VERSION   = 7
RF_FORMAT    = "yolov8"

# --- DATA_SOURCE = "kaggle-dataset" ---------------------------------
# Kaggle mounts a dataset under /kaggle/input/datasets/<owner>/<slug>.
ATTACHED_DATA_YAML = "/kaggle/input/datasets/zhalehmanbari/ir-card-yolo/data.yaml"

# Whether to flatten the cards before training on them.
#
# Off, and it has to stay off for this dataset: the cards fill about 45% of
# each frame with no clean border, and the classical rectifier finds an outline
# in only 12% of them. Rectifying would warp that 12% and merely resize the
# rest, training the model on two different geometries.
#
# This must match KYC_OCR_RECTIFY on the serving side. Train and serve have to
# agree on the geometry or the model sees a distribution it never saw.
RECTIFY = False

PROFILE = "ir_national_card_front"
BASE    = "yolov8n.pt"       # n is enough once the card is rectified
EPOCHS  = 150
IMGSZ   = 640
BATCH   = 16

# What to do when the assigned GPU is one this torch cannot drive.
#
# Kaggle still hands out Tesla P100s (sm_60), and the torch it preinstalls
# supports sm_70 and up - so on a P100 the GPU is simply unusable. The API
# cannot pick the accelerator type; only Settings > Accelerator can.
#
#   "cpu"  - train on CPU anyway. For 187 rectified cards that is roughly two
#            hours instead of ten minutes, but it needs nothing from you.
#   "fail" - stop, so you can switch the accelerator to T4 and re-run.
ON_UNUSABLE_GPU = "cpu"

# Source of the kyc package. scripts/kaggle_run.py uploads it as a private
# dataset and attaches it here.
SRC_DATASET = "/kaggle/input/kyc-src"

# Falling back to a git clone is opt-in. It runs whatever is on the repo's
# default branch, which is not necessarily the code you just pushed.
ALLOW_GIT_CLONE = False
REPO_URL        = "https://github.com/zhaleh197/kyc.git"


## 2. Dependencies

In [ ]:
# --no-deps on ultralytics is load-bearing, not an optimisation.
#
# Installing it normally pulls a newer torch, and recent torch wheels have
# dropped sm_60. Kaggle still hands out Tesla P100s (sm_60), so the upgrade
# silently leaves CUDA unusable: torch imports, cuda.is_available() may even be
# True, and every kernel launch then fails. Keeping Kaggle's own torch avoids
# the whole problem.
#
# pydantic-settings is needed because the training script imports the document
# profile from the kyc package - that is the point, so the class order used
# here cannot drift from the one the serving pipeline expects.
!pip install -q --no-deps ultralytics
!pip install -q ultralytics-thop onnx onnxsim pydantic-settings roboflow

import torch
import ultralytics

print("ultralytics", ultralytics.__version__)
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

# DEVICE is what gets handed to ultralytics later. Deciding it here, once,
# keeps the training cell from having to re-derive it.
DEVICE = "cpu"
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    supported = torch.cuda.get_arch_list()
    print(f"gpu: {torch.cuda.get_device_name(0)}  sm_{major}{minor}")
    print("torch was built for:", " ".join(supported))

    if f"sm_{major}{minor}" in supported:
        DEVICE = "0"
    elif ON_UNUSABLE_GPU == "cpu":
        # cuda.is_available() is True here and every kernel launch would still
        # fail, so this has to be caught by capability, not availability.
        print(
            f"\nWARNING: this torch cannot drive sm_{major}{minor}. Falling back to CPU.\n"
            "         Expect roughly two hours instead of ten minutes.\n"
            "         For a fast run: Settings > Accelerator > GPU T4 x2, then re-run."
        )
    else:
        raise SystemExit(
            f"Assigned GPU is sm_{major}{minor}, which this torch cannot drive.\n"
            "Switch Settings > Accelerator to GPU T4 x2 and re-run, or set "
            'ON_UNUSABLE_GPU = "cpu" to train on CPU anyway.'
        )
else:
    print("no GPU visible - training on CPU")

print("device:", DEVICE)


## 3. Get the source

The training script and the document profiles come from the repo, so nothing
about the card layout is duplicated into this notebook.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

WORK = Path("/kaggle/working/src")
INPUT = Path("/kaggle/input")


def find_source_tree(root):
    """Locate the uploaded package instead of assuming where Kaggle put it.

    Kaggle does not always mount a dataset at /kaggle/input/<slug>: a run of
    this notebook found everything under /kaggle/input/datasets instead, and
    the hard-coded path turned a perfectly good upload into a failed run.
    Searching for a marker file works whatever the layout turns out to be.
    """
    if not root.exists():
        return None
    for marker in root.rglob("scripts/roboflow_pull.py"):
        candidate = marker.parent.parent
        if (candidate / "kyc" / "__init__.py").exists():
            return candidate
    return None


print("attached under /kaggle/input:")
for entry in sorted(INPUT.rglob("*")) if INPUT.exists() else []:
    if entry.is_dir() and len(entry.relative_to(INPUT).parts) <= 2:
        print("   ", entry.relative_to(INPUT))

source_tree = Path(SRC_DATASET) if Path(SRC_DATASET).exists() else find_source_tree(INPUT)

if source_tree is not None:
    print("\nusing source tree:", source_tree)
    if WORK.exists():
        shutil.rmtree(WORK)
    shutil.copytree(source_tree, WORK)
elif ALLOW_GIT_CLONE:
    # Opt-in only. Falling back to the published repo silently is worse than
    # failing: it runs whatever code happens to be on the default branch, which
    # is how a run once failed on a module that only existed locally.
    print("\nsource dataset missing; cloning", REPO_URL)
    if not WORK.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORK)], check=True)
else:
    raise SystemExit(
        f"No source tree found under /kaggle/input - looked for {SRC_DATASET} and "
        "for any directory containing scripts/roboflow_pull.py.\n"
        "Push it with:  python -m scripts.kaggle_run push\n"
        "Or set ALLOW_GIT_CLONE = True to run whatever is on the repo's default "
        "branch instead - only do that if you know it is current."
    )

os.chdir(WORK)
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
print("cwd:", os.getcwd())


PROGRESS = Path("/kaggle/working/progress")
PROGRESS.mkdir(parents=True, exist_ok=True)


def mark(step, detail=""):
    """Leave a breadcrumb in the output directory.

    The kernel's execution log is not always reachable - on a restricted
    network the log endpoint can be blocked while the output file *listing*
    still comes through. These markers make that bare listing enough to tell
    how far the run got.
    """
    path = PROGRESS / f"{step}.txt"
    path.write_text(str(detail), encoding="utf-8")
    print(f"[progress] {step} {detail}".rstrip(), flush=True)


def run(*args):
    """Run a step and stream its output.

    check=True alone would raise CalledProcessError showing only the command,
    while the child's diagnostics - which class had no annotations, how many
    cards could not be rectified - vanish. In Jupyter a subprocess inheriting
    the parent's stdout writes to the kernel's real stdout, which does not
    reach the cell output either. So the output is piped and echoed here.
    """
    label = args[1] if args and args[0] == "-m" else (args[0] if args else "step")
    print(">", " ".join(str(a) for a in args), flush=True)
    process = subprocess.Popen(
        [sys.executable, *args],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if code != 0:
        raise SystemExit(f"{label} exited with code {code} - see the output above")
    mark(f"ok-{Path(label).stem}")


mark("01-source-ready", os.getcwd())


## 4. Get the data

With `DATA_SOURCE = "roboflow"` the dataset is pulled here, inside the kernel.
The API key comes from a Kaggle Secret, never from this notebook - a key
committed into a notebook is exactly how the previous one leaked.


In [ ]:
import os
from pathlib import Path

RAW = Path("/kaggle/working/data/raw")
NORMALISED = Path("/kaggle/working/data/normalised")
RECTIFIED = Path("/kaggle/working/data/rectified")

if DATA_SOURCE == "roboflow":
    try:
        from kaggle_secrets import UserSecretsClient

        os.environ["ROBOFLOW_API_KEY"] = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
        print("Roboflow key loaded from Kaggle Secrets")
    except Exception as exc:
        raise SystemExit(
            "Could not read the ROBOFLOW_API_KEY secret: " + str(exc) + "\n"
            "Add it under Add-ons > Secrets, then re-run this cell."
        ) from exc

    run("-m", "scripts.roboflow_pull",
        "--workspace", RF_WORKSPACE,
        "--project", RF_PROJECT,
        "--version", str(RF_VERSION),
        "--format", RF_FORMAT,
        "--profile", PROFILE,
        "--raw", str(RAW),
        "--out", str(NORMALISED))
    mark("02-roboflow-pulled")
    dataset_dir = NORMALISED
else:
    attached = Path(ATTACHED_DATA_YAML).parent
    if not attached.is_dir():
        # Same lesson as the source tree: Kaggle's mount layout is not worth
        # hard-coding. Find the dataset by its shape instead.
        found = [
            y.parent
            for y in Path("/kaggle/input").rglob("data.yaml")
            if (y.parent / "train").is_dir() and "kyc-src" not in str(y)
        ]
        if not found:
            raise SystemExit(
                f"No dataset at {ATTACHED_DATA_YAML}, and no other data.yaml with a train/ "
                "folder under /kaggle/input. Attach it under Add Data, or upload it with "
                "`python -m scripts.kaggle_run upload-data`."
            )
        attached = found[0]
    print("using attached dataset at", attached)
    # A hand-downloaded Roboflow export carries Roboflow's class order, which
    # is almost never the profile's. Remapping here means the manual route gets
    # the same guarantee as the API route - otherwise the model would train on
    # correctly-drawn boxes carrying the wrong labels, and nothing downstream
    # would notice.
    run("-m", "scripts.roboflow_pull",
        "--skip-download",
        "--profile", PROFILE,
        "--raw", str(attached),
        "--out", str(NORMALISED))
    mark("02-attached-normalised")
    dataset_dir = NORMALISED

if RECTIFY:
    run("-m", "scripts.rectify_dataset",
        "--src", str(dataset_dir),
        "--out", str(RECTIFIED),
        "--profile", PROFILE)
    mark("03-rectified")
    DATA_YAML = str(RECTIFIED / "data.yaml")
else:
    DATA_YAML = str(dataset_dir / "data.yaml")

print("\nDATA_YAML =", DATA_YAML)


## 5. Fail fast on a class-order mismatch

A reordered class list is silent and catastrophic: the pipeline would write the
father's name into the national id field and every downstream check would still
pass. Better to find out now than after a long training run.


In [ ]:
import yaml

from kyc.modules.ocr.profiles import get_profile

profile = get_profile(PROFILE)
expected = list(profile.class_names)

with open(DATA_YAML, encoding="utf-8") as handle:
    spec = yaml.safe_load(handle)

names = spec["names"]
actual = [names[i] for i in sorted(names)] if isinstance(names, dict) else list(names)

print("profile :", expected)
print("dataset :", actual)

if actual != expected:
    raise SystemExit(
        "Class order mismatch. Fix the names: list in data.yaml to match the "
        "profile exactly, then re-run this cell."
    )
print()
print("OK - class order matches. Model will be saved as:", profile.detector_model)
mark("04-class-order-ok")


## 6. Train and export

In [ ]:
# run() streams the child's output; a bare subprocess.run would hide the
# training log behind a CalledProcessError.
run("training/kaggle/train_field_detector.py",
    "--data", DATA_YAML,
    "--profile", PROFILE,
    "--model", BASE,
    "--epochs", str(EPOCHS),
    "--imgsz", str(IMGSZ),
    "--batch", str(BATCH),
    "--device", DEVICE,
    "--out", "/kaggle/working/models")
mark("05-trained")


## 7. Verify the artifact

In [ ]:
import hashlib
from pathlib import Path

out = Path("/kaggle/working/models")
for path in sorted(out.glob("*")):
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f"{path.name:<40} {path.stat().st_size/1e6:7.1f} MB  sha256:{digest}")

print()
print("Record the sha256 in models/manifest.yaml with the dataset and commit.")
print("Then, in the repo:  python -m scripts.check_models")


## 8. Next: the text recognisers

The detector alone makes the pipeline work with the pretrained `easyocr`
backend. To go faster and more accurate, bootstrap a labelled crop set from
easyocr output and train the CRNNs - see `training/kaggle/README.md`.
